In [ ]:
import os

# W&B 로깅을 완전히 비활성화합니다.
os.environ['WANDB_MODE'] = 'disabled'

In [ ]:
!pip install torch torchvision transformers
!pip install accelerate peft
!pip install pillow requests
!pip install -U bitsandbytes

In [ ]:
import torch
import gc

# GPU 캐시 비우기
torch.cuda.empty_cache()

# 가비지 컬렉션 실행
gc.collect()

# VRAM 사용량 확인
print(f"할당된 메모리: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print(f"예약된 메모리: {torch.cuda.memory_reserved()/1024**3:.2f} GB")

할당된 메모리: 0.00 GB
예약된 메모리: 0.00 GB


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## **2차 파인튜닝 코드**

In [ ]:
!pip install -U "transformers>=4.40.0" accelerate safetensors datasets pillow

import os
import json
from dataclasses import dataclass
from typing import Dict, List, Any

import torch
from torch.utils.data import Dataset
from PIL import Image

from transformers import (
    AutoProcessor,
    LlavaForConditionalGeneration,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
)

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,              # ✅ 4bit → 8bit 양자화
    llm_int8_threshold=3.0,
    llm_int8_has_fp16_weight=False
)

In [ ]:
# 1차 SFT까지 끝난 체크포인트 경로 (너 환경에 맞게 수정)
SFT_CHECKPOINT_DIR = "/content/drive/MyDrive/dataset/finetuned_test_04"
BASE_MODEL_ID = "llava-hf/llava-1.5-7b-hf"

device = "cuda" if torch.cuda.is_available() else "cpu"

# processor는 SFT 체크포인트나 base 중 아무거나 써도 됨 (토크나이저 구조 동일하면 OK)
processor = AutoProcessor.from_pretrained(SFT_CHECKPOINT_DIR)

model = LlavaForConditionalGeneration.from_pretrained(
    SFT_CHECKPOINT_DIR,
    quantization_config=bnb_config,
    device_map="auto",
)

model.config.use_cache = False
print("✅ 8bit SFT 모델 로드 완료")

# 🔥🔥🔥 PEFT (LoRA) 설정 추가 🔥🔥🔥
# 1. k-bit 학습을 위해 모델을 준비 (8bit)
model = prepare_model_for_kbit_training(model)

# 2. LoRA 설정 정의 (R: 64, Alpha: 16은 흔히 사용되는 최적 설정)
lora_config = LoraConfig(
    r=64,
    lora_alpha=16,
    # LLaVA 1.5에서 Q, K, V, O (Attention Projection) 레이어에 LoRA 적용
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
)

# 3. 모델에 LoRA 어댑터 적용
model = get_peft_model(model, lora_config)
# 학습 가능한 파라미터 수를 확인하여 메모리 효율성 확인
model.print_trainable_parameters()
# 🔥🔥🔥 LoRA 설정 끝 🔥🔥🔥

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


config.json:   0%|          | 0.00/950 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

✅ 8bit SFT 모델 로드 완료


/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:282: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 100,902,912 || all params: 7,164,329,984 || trainable%: 1.4084


In [ ]:
import json
import random
from typing import Dict, List, Any
from torch.utils.data import Dataset

class DermaRaftTextDatasetWithDocs(Dataset):
    """
    RAFT 텍스트 전용 데이터셋
    - query
    - golden (label, text)
    - hard_negatives: list of {label, text}
    - easy_negative: {label, text} or None

    __getitem__에서 매번 문맥 순서를 랜덤으로 섞고,
    그 중 어떤 번호가 golden인지 golden_doc_id로 반환.
    """

    def __init__(self, path: str):
        self.samples: List[Dict[str, Any]] = []

        # JSON 배열 vs JSONL 자동 감지
        with open(path, "r", encoding="utf-8") as f:
            first_char = f.read(1)
            f.seek(0)
            if first_char == "[":
                raw_list = json.load(f)
            else:
                raw_list = []
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    raw_list.append(json.loads(line))

        for obj in raw_list:
            query     = obj.get("query")
            golden    = obj.get("golden", {}) or {}
            hard_negs = obj.get("hard_negatives", []) or []
            easy_neg  = obj.get("easy_negative", None)

            golden_text = golden.get("text")
            golden_label = golden.get("label")

            if not query or not golden_text:
                # 필수 정보 없으면 스킵
                continue

            # raw 형태로 저장해두고, 실제 문맥 배열은 __getitem__에서 섞어서 생성
            self.samples.append({
                "query": query,
                "golden_text": golden_text,
                "golden_label": golden_label,
                "hard_negatives": hard_negs,
                "easy_negative": easy_neg,
            })

        print(f"✅ RAFT 텍스트+문맥 데이터 로드 완료: {len(self.samples)} samples")
        if len(self.samples) > 0:
            print("  예시 1개 (raw):", self.samples[0])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        base = self.samples[idx]

        query         = base["query"]
        golden_text   = base["golden_text"]
        golden_label  = base["golden_label"]
        hard_negs     = base["hard_negatives"]
        easy_neg      = base["easy_negative"]

        # (role, text) 리스트로 모으기
        docs_raw = [("golden", golden_text)]
        for hn in hard_negs:
            docs_raw.append(("hard_negative", hn.get("text")))
        if easy_neg is not None:
            docs_raw.append(("easy_negative", easy_neg.get("text")))

        # 텍스트 없는 것 제거
        docs_raw = [(role, txt) for role, txt in docs_raw if txt]

        # 매 샘플마다 문맥 순서를 랜덤 셔플
        random.shuffle(docs_raw)

        docs_with_id = []
        golden_doc_id = None
        for i, (role, txt) in enumerate(docs_raw, start=1):
            docs_with_id.append({
                "id": i,
                "role": role,
                "text": txt,
            })
            if role == "golden":
                golden_doc_id = i

        # golden_doc_id는 반드시 하나 있어야 함
        assert golden_doc_id is not None, "golden 문맥이 누락된 샘플이 있습니다."

        return {
            "question": query,
            "answer": golden_text,       # 실제 최종 질의 답변 텍스트
            "label": golden_label,
            "docs": docs_with_id,        # 섞인 문맥 리스트
            "golden_doc_id": golden_doc_id,
        }

In [ ]:
RAFT_TRAIN_JSONL_PATH = "/content/raft_train_dataset_final.jsonl"
train_dataset = DermaRaftTextDatasetWithDocs(RAFT_TRAIN_JSONL_PATH)

✅ RAFT 텍스트+문맥 데이터 로드 완료: 4240 samples
  예시 1개 (raw): {'query': '여드름이 생기는 가장 기본적인 이유가 궁금해요. 호르몬, 피지, 세균이 서로 어떻게 작용해서 여드름이 되는 건가요?', 'golden_text': '여드름은 모낭(모발이 자라는 피부의 구멍)의 염증을 일으키는 호르몬, 피지, 세균 간의 상호작용에 의해 발생합니다. 여드름은 많은 유형의 피부 이상(병변)을 특징으로 합니다. 이들은 크기와 중증도에서 다르고, 일부는 다른 성장물보다 피부 깊숙이 침범합니다. 블랙헤드(열린 면포) 패립종(닫힌 면포) 뾰루지(염증성 닫힌 면포) 융기된 딱딱한 돌기(구진) 고름이 든 표면 돌기(농포) 고름이 든 보다 깊고 단단한 돌기(결절) 고름이 된 보다 큰 주머니(낭) 때때로 고름이 든 훨씬 크고 깊은 주머니(농양) 낭과 농양은 모두 고름이 찬 주머니이지만 농양이 좀 더 크고 뿌리가 깊은 편입니다. 지방성 물질(피지)을 분비하는 피지선은 피부의 중간층인 진피층에 위치합니다. 이러한 분비선은 모낭에 붙어 있습니다. 피지는 죽은 피부 세포와 함께 피지선과 모낭을 통과하여 모공을 통해 피부 표면으로 배출됩니다. 건조된 피지, 죽은 피부 세포, 세균 등이 모여 모낭을 막아 피지가 모공을 통해 배출되는 것을 차단하게 되면 여드름이 발생합니다. 완전히 차단되지 않은 경우에는 블랙헤드(열린 면포)가 생깁니다. 완전히 차단되면 패립종(닫힌 면포)이 발생합니다. 뾰루지는 염증성 패립종입니다. 모낭이 피지로 가득 차서 막히면 모낭에서 흔히 발견되는 세균인 모낭관 내 여드름균(이전 여드름 유발균)의 과다증식을 촉진합니다', 'golden_label': 'Acne', 'hard_negatives': [{'source': '강남세브란스병원', 'label': 'Rosacea', 'title': '주사피부염의 레이저 치료', 'text': '혈관 병변에는 레이저 치료가 효과적입니다. 펄스다이레이저(PDL)는 모세혈관 확장과 홍반 치료에 가장 많이

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Any
import torch

@dataclass
class DataCollatorRaftWithDocs:
    processor: Any

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        prompts: List[str] = []
        targets: List[str] = []

        for f in features:
            q = f["question"]
            answer_text = f["answer"]
            docs = f["docs"]
            golden_doc_id = f["golden_doc_id"]

            # 문맥 블록 문자열 구성
            # [1] ..., [2] ..., ...
            docs_lines = []
            for d in docs:
                docs_lines.append(f"[{d['id']}] {d['text']}")
            docs_block = "\n".join(docs_lines)

            # USER 프롬프트 구성 (한국어 설명 포함)
            prompt_text = (
                f"{self.processor.tokenizer.bos_token}"
                "USER:\n"
                f"질문: {q}\n\n"
                "아래는 이 질문과 관련 있을 수도 있고 없을 수도 있는 문맥들이다.\n"
                "각 문맥은 [번호]로 표시되어 있다.\n\n"
                f"{docs_block}\n\n"
                "위 문맥들 중에서 질문에 가장 적절한 문맥 번호 하나를 고르고,\n"
                "그 문맥을 근거로 답변하라.\n"
                "반드시 아래 형식을 따르라.\n\n"
                "형식:\n"
                "근거 문맥: [번호]\n"
                "답변: (질문에 대한 자세한 설명)\n\n"
                "ASSISTANT:"
            )

            # 정답 텍스트 (golden 문맥 번호 + golden_text)
            target_text = (
                f"근거 문맥: [{golden_doc_id}]\n"
                f"답변: {answer_text.strip()}"
            )

            prompts.append(prompt_text)
            targets.append(target_text)

        # 1) prompt만 인코딩
        enc_prompt = self.processor(
            text=prompts,
            padding="longest",
            return_tensors="pt"
        )

        # 2) prompt + target 같이 인코딩
        full_texts = [p + " " + t for p, t in zip(prompts, targets)]
        enc_full = self.processor(
            text=full_texts,
            padding="longest",
            return_tensors="pt"
        )

        input_ids = enc_full["input_ids"]
        attention_mask = enc_full["attention_mask"]

        labels = input_ids.clone()
        pad_id = self.processor.tokenizer.pad_token_id

        # prompt 부분은 loss에서 제외 (-100)
        for i in range(len(prompts)):
            prompt_ids = enc_prompt["input_ids"][i]
            prompt_len = (prompt_ids != pad_id).sum()
            labels[i, :prompt_len] = -100

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }

In [ ]:
data_collator = DataCollatorRaftWithDocs(processor=processor)

In [ ]:
RAFT_JSONL_PATH = "/content/raft_train_dataset_final.jsonl"
output_dir = "/content/llava_raft_stage4_raftdocs"

train_dataset = DermaRaftTextDatasetWithDocs(RAFT_JSONL_PATH)
data_collator = DataCollatorRaftWithDocs(processor)

use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=6e-6,
    warmup_ratio=0.05,
    logging_steps=10,
    save_steps=200,
    save_total_limit=2,
    bf16=use_bf16,
    fp16=not use_bf16,
    gradient_checkpointing=True,
    remove_unused_columns=False,
    report_to="none",
)

✅ RAFT 텍스트+문맥 데이터 로드 완료: 4240 samples
  예시 1개 (raw): {'query': '여드름이 생기는 가장 기본적인 이유가 궁금해요. 호르몬, 피지, 세균이 서로 어떻게 작용해서 여드름이 되는 건가요?', 'golden_text': '여드름은 모낭(모발이 자라는 피부의 구멍)의 염증을 일으키는 호르몬, 피지, 세균 간의 상호작용에 의해 발생합니다. 여드름은 많은 유형의 피부 이상(병변)을 특징으로 합니다. 이들은 크기와 중증도에서 다르고, 일부는 다른 성장물보다 피부 깊숙이 침범합니다. 블랙헤드(열린 면포) 패립종(닫힌 면포) 뾰루지(염증성 닫힌 면포) 융기된 딱딱한 돌기(구진) 고름이 든 표면 돌기(농포) 고름이 든 보다 깊고 단단한 돌기(결절) 고름이 된 보다 큰 주머니(낭) 때때로 고름이 든 훨씬 크고 깊은 주머니(농양) 낭과 농양은 모두 고름이 찬 주머니이지만 농양이 좀 더 크고 뿌리가 깊은 편입니다. 지방성 물질(피지)을 분비하는 피지선은 피부의 중간층인 진피층에 위치합니다. 이러한 분비선은 모낭에 붙어 있습니다. 피지는 죽은 피부 세포와 함께 피지선과 모낭을 통과하여 모공을 통해 피부 표면으로 배출됩니다. 건조된 피지, 죽은 피부 세포, 세균 등이 모여 모낭을 막아 피지가 모공을 통해 배출되는 것을 차단하게 되면 여드름이 발생합니다. 완전히 차단되지 않은 경우에는 블랙헤드(열린 면포)가 생깁니다. 완전히 차단되면 패립종(닫힌 면포)이 발생합니다. 뾰루지는 염증성 패립종입니다. 모낭이 피지로 가득 차서 막히면 모낭에서 흔히 발견되는 세균인 모낭관 내 여드름균(이전 여드름 유발균)의 과다증식을 촉진합니다', 'golden_label': 'Acne', 'hard_negatives': [{'source': '강남세브란스병원', 'label': 'Rosacea', 'title': '주사피부염의 레이저 치료', 'text': '혈관 병변에는 레이저 치료가 효과적입니다. 펄스다이레이저(PDL)는 모세혈관 확장과 홍반 치료에 가장 많이

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

trainer.train()

trainer.save_model(output_dir)
processor.save_pretrained(output_dir)

print("✅ RAFT 8bit 2차 파인튜닝 완료:", output_dir)

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:181: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:181: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss
10,0.141600
20,0.106500
30,0.102100
40,0.235100
50,0.232000
60,0.337400
70,0.127300
80,0.167600
90,0.110900
100,0.042000


/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:181: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:181: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:181: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:181: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during

✅ RAFT 8bit 2차 파인튜닝 완료: /content/llava_raft_stage4_raftdocs


In [ ]:
# 모델이 저장된 폴더가 /content/trained_model 이라고 가정
# Google Drive의 '내 드라이브' 안에 'Saved_Models'라는 폴더를 만들고 그곳에 복사
# 드라이브에 백업
!cp -r "/content/llava_raft_stage4_raftdocs" "/content/drive/MyDrive/dataset/finetuned_test_RAFT_03"

## **엔드포인트**

In [ ]:
from transformers import AutoModelForVision2Seq, AutoProcessor
from peft import PeftModel

BASE_MODEL = "llava-hf/llava-1.5-7b-hf"          # 파인튜닝에 썼던 베이스
LORA_DIR   = "/content/llava_raft_stage4_raftdocs"  # LoRA 저장된 폴더
MERGED_DIR = "/content/llava_raft_merged"       # 합친 모델 저장할 위치

# 1) Base LLaVA 모델 로드
base = AutoModelForVision2Seq.from_pretrained(
    BASE_MODEL,
    torch_dtype="float16",
    low_cpu_mem_usage=True
)

# 2) LoRA 어댑터 로드 + 결합
model = PeftModel.from_pretrained(
    base,
    LORA_DIR,
    torch_dtype="float16"
)

# 3) LoRA를 base에 merge
model = model.merge_and_unload()

# 4) 합쳐진 모델 저장
model.save_pretrained(MERGED_DIR)

# 5) processor/tokenizer도 같이 복사
processor = AutoProcessor.from_pretrained(BASE_MODEL)
processor.save_pretrained(MERGED_DIR)

print("✅ Merge 완료! 저장 경로:", MERGED_DIR)

/usr/local/lib/python3.12/dist-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

processor_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/674 [00:00<?, ?B/s]

...

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/41.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/552 [00:00<?, ?B/s]

✅ Merge 완료! 저장 경로: /content/llava_raft_merged


In [ ]:
from huggingface_hub import HfApi, upload_folder

HF_TOKEN = "<YOUR_HF_TOKEN>"  # ⚠️ 아래 참고
REPO_ID = "jayun/llava_raft_01"
MODEL_DIR = "/content/llava_raft_merged"

api = HfApi(token=HF_TOKEN)

api.create_repo(
    repo_id=REPO_ID,
    private=True,
    exist_ok=True
)

upload_folder(
    folder_path=MODEL_DIR,
    repo_id=REPO_ID,
    token=HF_TOKEN,
    commit_message="Upload merged LLaVA RAFT model"
)

print("✅ 업로드 완료:", f"https://huggingface.co/{REPO_ID}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ft_merged/tokenizer.model: 100%|##########|  500kB /  500kB            

  ...0002-of-00003.safetensors:   1%|          | 41.9MB / 4.96GB            

  ...0001-of-00003.safetensors:   1%|          | 41.9MB / 4.99GB            

  ...0003-of-00003.safetensors:   1%|1         | 41.9MB / 4.18GB            

✅ 업로드 완료: https://huggingface.co/jayun/llava_raft_01_merged


In [ ]:
import os

MERGED_DIR = "/content/llava_raft_merged"

# 저장된 파일 확인
print("📁 저장된 파일 목록:")
for root, dirs, files in os.walk(MERGED_DIR):
    level = root.replace(MERGED_DIR, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files:
        filepath = os.path.join(root, file)
        size_mb = os.path.getsize(filepath) / (1024*1024)
        print(f'{subindent}{file} ({size_mb:.2f} MB)')

📁 저장된 파일 목록:
llava_raft_merged/
  model-00002-of-00003.safetensors (4728.20 MB)
  chat_template.jinja (0.00 MB)
  model.safetensors.index.json (0.07 MB)
  tokenizer_config.json (0.00 MB)
  tokenizer.model (0.48 MB)
  processor_config.json (0.00 MB)
  config.json (0.00 MB)
  generation_config.json (0.00 MB)
  preprocessor_config.json (0.00 MB)
  tokenizer.json (3.45 MB)
  special_tokens_map.json (0.00 MB)
  model-00001-of-00003.safetensors (4761.63 MB)
  added_tokens.json (0.00 MB)
  model-00003-of-00003.safetensors (3982.67 MB)


In [ ]:
from huggingface_hub import HfApi
import os

HF_TOKEN = "<YOUR_HF_TOKEN>"
REPO_ID = "jayun/llava_raft_01"
MERGED_DIR = "/content/llava_raft_merged"

print("🚀 Hugging Face 업로드 시작...")
print(f"📁 업로드 경로: {MERGED_DIR}")
print(f"🎯 대상 Repo: {REPO_ID}")
print("-" * 60)

# README 생성 (없으면)
readme_path = os.path.join(MERGED_DIR, "README.md")
if not os.path.exists(readme_path):
    readme_content = """---
license: llama2
base_model: llava-hf/llava-1.5-7b-hf
tags:
- llava
- vision
- multimodal
- raft
- fine-tuned
language:
- en
pipeline_tag: image-text-to-text
---

# LLaVA 1.5 7B - RAFT Fine-tuned

이 모델은 `llava-hf/llava-1.5-7b-hf`를 RAFT (Retrieval Augmented Fine-Tuning) 기법으로 파인튜닝한 비전-언어 모델입니다.

## 모델 정보
- **Base Model**: llava-hf/llava-1.5-7b-hf
- **Model Size**: 7B parameters
- **Fine-tuning Method**: RAFT
- **Training Stages**: 2-stage fine-tuning

## 사용 방법
```python
from transformers import AutoModelForVision2Seq, AutoProcessor
from PIL import Image
import torch

# 모델 로드
model = AutoModelForVision2Seq.from_pretrained(
    "jayun/llava_raft_01",
    torch_dtype=torch.float16,
    device_map="auto"
)
processor = AutoProcessor.from_pretrained("jayun/llava_raft_01")

# 이미지 준비
image = Image.open("your_image.jpg")
prompt = "USER: <image>\\nDescribe this image in detail.\\nASSISTANT:"

# 추론
inputs = processor(text=prompt, images=image, return_tensors="pt").to("cuda", torch.float16)
output = model.generate(**inputs, max_new_tokens=200, do_sample=False)
response = processor.decode(output[0], skip_special_tokens=True)
print(response)
```

## 모델 구조
- Vision Encoder: CLIP ViT-L/14
- Language Model: Vicuna-7B
- Total Parameters: ~7B
- Precision: FP16

## 학습 데이터
- Stage 1: 기본 파인튜닝
- Stage 2: RAFT documents를 활용한 추가 학습

## 제한사항
- 영어 위주로 학습됨
- 이미지 해상도: 336x336 권장
- GPU 메모리: 최소 16GB 필요
"""
    with open(readme_path, "w", encoding="utf-8") as f:
        f.write(readme_content)
    print("✅ README.md 생성 완료")

api = HfApi(token=HF_TOKEN)

# Repo 생성 (이미 있으면 무시)
try:
    api.create_repo(
        repo_id=REPO_ID,
        private=True,  # private으로 할지 public으로 할지 선택
        exist_ok=True
    )
    print(f"✅ Repo 확인/생성: {REPO_ID}")
except Exception as e:
    print(f"⚠️  Repo 생성 중 경고: {e}")

# 업로드 (대용량 파일이므로 시간이 걸립니다)
print("\n📤 파일 업로드 중... (13GB 정도라 10-30분 소요될 수 있습니다)")
print("   진행상황은 터미널에서 확인하세요.")

try:
    api.upload_folder(
        folder_path=MERGED_DIR,
        repo_id=REPO_ID,
        commit_message="Upload fully merged LLaVA RAFT model (13.5GB)",
        ignore_patterns=[".git/*", "*.pyc", "__pycache__/*", ".DS_Store"]
    )

    print("\n" + "=" * 60)
    print("✅ 업로드 완료!")
    print("=" * 60)
    print(f"🔗 모델 확인: https://huggingface.co/{REPO_ID}")
    print(f"🔗 파일 확인: https://huggingface.co/{REPO_ID}/tree/main")

except Exception as e:
    print(f"\n❌ 업로드 중 오류 발생:")
    print(f"   {str(e)}")
    print("\n💡 해결 방법:")
    print("   1. 인터넷 연결 확인")
    print("   2. HF 토큰 권한 확인 (write 권한 필요)")
    print("   3. Colab Pro 사용 시 더 안정적")

🚀 Hugging Face 업로드 시작...
📁 업로드 경로: /content/llava_raft_merged
🎯 대상 Repo: jayun/llava_raft_01
------------------------------------------------------------
✅ README.md 생성 완료
✅ Repo 확인/생성: jayun/llava_raft_01

📤 파일 업로드 중... (13GB 정도라 10-30분 소요될 수 있습니다)
   진행상황은 터미널에서 확인하세요.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ft_merged/tokenizer.model: 100%|##########|  500kB /  500kB            

  ...0001-of-00003.safetensors:   1%|          | 41.9MB / 4.99GB            

  ...0003-of-00003.safetensors:   1%|          | 33.5MB / 4.18GB            

  ...0002-of-00003.safetensors:   1%|          | 41.9MB / 4.96GB            


✅ 업로드 완료!
🔗 모델 확인: https://huggingface.co/jayun/llava_raft_01
🔗 파일 확인: https://huggingface.co/jayun/llava_raft_01/tree/main


In [ ]:
# Colab에서 실행
MERGED_DIR = "/content/llava_raft_merged"

# handler.py 내용을 위 코드로 작성
handler_code = '''
from typing import Dict, List, Any
from PIL import Image
import torch
import base64
from io import BytesIO

class EndpointHandler:
    def __init__(self, path=""):
        from transformers import AutoModelForVision2Seq, AutoProcessor

        self.model = AutoModelForVision2Seq.from_pretrained(
            path,
            torch_dtype=torch.float16,
            device_map="auto"
        )
        self.processor = AutoProcessor.from_pretrained(path)
        self.model.eval()

    def __call__(self, data: Dict[str, Any]) -> List[Dict[str, Any]]:
        try:
            inputs = data.get("inputs", {})

            if isinstance(inputs.get("image"), str):
                image_data = base64.b64decode(inputs["image"])
                image = Image.open(BytesIO(image_data)).convert("RGB")
            else:
                image = inputs["image"]

            question = inputs.get("question", "Describe this image.")
            prompt = f"USER: <image>\\n{question}\\nASSISTANT:"

            model_inputs = self.processor(
                text=prompt,
                images=image,
                return_tensors="pt"
            ).to(self.model.device)

            with torch.no_grad():
                output = self.model.generate(
                    **model_inputs,
                    max_new_tokens=200,
                    do_sample=False
                )

            response = self.processor.decode(output[0], skip_special_tokens=True)

            if "ASSISTANT:" in response:
                response = response.split("ASSISTANT:")[-1].strip()

            return [{"generated_text": response}]

        except Exception as e:
            return [{"error": str(e)}]
'''

# handler.py 저장
with open(f"{MERGED_DIR}/handler.py", "w") as f:
    f.write(handler_code)

print("✅ handler.py 생성 완료")

# Hugging Face에 업로드
from huggingface_hub import HfApi

HF_TOKEN = "<YOUR_HF_TOKEN>"
REPO_ID = "jayun/llava_raft_01"

api = HfApi(token=HF_TOKEN)

# handler.py만 업로드
api.upload_file(
    path_or_fileobj=f"{MERGED_DIR}/handler.py",
    path_in_repo="handler.py",
    repo_id=REPO_ID,
    commit_message="Add custom handler for Inference Endpoint"
)

print(f"✅ handler.py 업로드 완료!")
print(f"🔗 확인: https://huggingface.co/{REPO_ID}/blob/main/handler.py")

✅ handler.py 생성 완료
✅ handler.py 업로드 완료!
🔗 확인: https://huggingface.co/jayun/llava_raft_01/blob/main/handler.py
